# Scenario 3: Customer Segmentation Query
**Scenario:** The product team needs to segment customers based on their purchasing behavior for a new feature rollout.

## Prompt
**Initial Prompt Strategy:**
> "I need a SQL query to segment customers based on three tables: `user_activity` (user_id, last_login_date, feature_usage_count, account_type), `transactions` (transaction_id, user_id, transaction_date, amount, platform), and `user_preferences` (user_id, communication_preference, interface_theme, notification_settings).
> 
> Please write a query to:
> 1. Find active users (logged in within the last 30 days).
> 2. Filter these down to high-value customers, defined as the top 20% by total spending.
> 3. Show the user preference trends (like communication preference and interface theme) for this specific segment of customers."

## Generated Code

In [ ]:
-- Initial SQL Generation (Using Subqueries and standard WHERE clauses)
SELECT 
    up.communication_preference,
    up.interface_theme,
    COUNT(up.user_id) AS customer_count
FROM user_preferences up
JOIN user_activity ua ON up.user_id = ua.user_id
JOIN (
    SELECT user_id, SUM(amount) as total_spent
    FROM transactions
    GROUP BY user_id
) t ON up.user_id = t.user_id
WHERE ua.last_login_date >= CURRENT_DATE - INTERVAL '30 days'
  AND t.user_id IN (
      SELECT user_id 
      FROM (
          SELECT user_id, 
                 NTILE(5) OVER (ORDER BY SUM(amount) DESC) as spending_tier
          FROM transactions
          GROUP BY user_id
      ) sub
      WHERE spending_tier = 1
  )
GROUP BY 
    up.communication_preference, 
    up.interface_theme
ORDER BY customer_count DESC;

## Follow-up Prompts
**Refinement & Optimization:**
The initial query works, but it's nested and recalculates the total spending twice (once for the join, once for the `NTILE` filter). The date filtering is also dialect-specific (`INTERVAL '30 days'`). 

> *"Can you optimize this SQL query to be more readable and performant?*
> *1. Use Common Table Expressions (CTEs) instead of nested subqueries.*
> *2. Use `PERCENT_RANK()` instead of `NTILE(5)` for calculating the top 20%, as it's often more statistically accurate for continuous spending values.*
> *3. Add a filter to ensure we only look at transactions that also occurred in the last year, so legacy spending doesn't skew the results."*

## Final Solution

In [ ]:
-- Optimized SQL Generation (Using CTEs and Percent_Rank)

WITH active_users AS (
    -- Identify users active in the last 30 days
    SELECT user_id
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL '30 days'
),
user_spending AS (
    -- Calculate total spending over the last year for active users
    SELECT 
        t.user_id, 
        SUM(t.amount) AS total_spent
    FROM transactions t
    INNER JOIN active_users au ON t.user_id = au.user_id
    WHERE t.transaction_date >= CURRENT_DATE - INTERVAL '1 year'
    GROUP BY t.user_id
),
spending_percentiles AS (
    -- Calculate the spending percentile (Top 20% means percentile >= 0.80)
    SELECT 
        user_id,
        PERCENT_RANK() OVER (ORDER BY total_spent ASC) AS spend_percentile
    FROM user_spending
),
high_value_users AS (
    -- Filter down to the top 20%
    SELECT user_id
    FROM spending_percentiles
    WHERE spend_percentile >= 0.80
)

-- Final Aggregation: Analyze preference trends for this target segment
SELECT 
    up.communication_preference,
    up.interface_theme,
    up.notification_settings,
    COUNT(up.user_id) AS total_users,
    ROUND(COUNT(up.user_id) * 100.0 / SUM(COUNT(up.user_id)) OVER(), 2) AS percentage_of_segment
FROM user_preferences up
INNER JOIN high_value_users hvu ON up.user_id = hvu.user_id
GROUP BY 
    up.communication_preference,
    up.interface_theme,
    up.notification_settings
ORDER BY total_users DESC;

### Discussion Points:

**1. How were percentile calculations handled?**
* **Initial Approach:** The AI used the `NTILE(5)` window function and filtered `WHERE spending_tier = 1`. This essentially divides the dataset into 5 equal buckets. While functional, `NTILE` forces exactly equal bucket sizes, which can artificially split users who spent the exact same amount into different tiers.
* **Refined Approach:** The prompt specifically requested `PERCENT_RANK()`. This assigns a continuous rank between 0 and 1, allowing us to filter `WHERE spend_percentile >= 0.80` (the top 20%). This handles ties perfectly and is much more accurate for monetary distributions.

**2. What approaches to date filtering were used?**
* Both iterations used standard PostgreSQL/MySQL date arithmetic (`CURRENT_DATE - INTERVAL '30 days'`). However, a major difference was *where* the filter was applied.
* In the follow-up prompt, a constraint was added to filter the `transactions` table by the last year (`INTERVAL '1 year'`). Without this constraint, a user who spent a massive amount 5 years ago but only logged in to check something yesterday would be incorrectly classified as a "high-value" target for a new feature.

**3. How was the query optimized, CTE, subquery etc…?**
* **Subqueries (Initial):** The first query used nested `SELECT` statements in the `FROM` and `WHERE IN (...)` clauses. This was hard to read and highly unoptimized because it calculated total spending for the *entire* user base twice.
* **CTEs (Refined):** The final solution utilized `WITH` clauses (Common Table Expressions). It created a logical, step-by-step pipeline: find active users -> calculate spending *only* for active users -> rank them -> extract preferences. This drastically reduces the dataset size at each join, greatly improving database performance and making the query significantly easier for human engineers to review.